# CVFEST-style Descriptor Projection And Stiffness Intervention Screen

This notebook starts after CV discovery. It focuses the descriptor projection on final p-ratio and y-compression, then screens simple stiffness interventions suggested by the descriptor model. Final p-ratio remains the main functional outcome; y-compression is the deformation target; x-compression is intentionally left out of the main analysis.

By default it trains the CV2 model directly on `new_reid_combined.pt`, then projects final p-ratio and y-compression onto stiffness/edge/Hessian descriptors.


In [1]:
from __future__ import annotations

from pathlib import Path
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj.to_string(index=False) if hasattr(obj, 'to_string') else obj)
import torch.nn.functional as F

from graph_utils import calc_p_ratio_rollout_sides
from course_project.data import load_dataset, resolve_dataset_splits
from course_project.graph import clone_graph
from course_project.hessian import make_r0_dict, build_spring_hessian_2d
from course_project.models.latent_space_simulator import LatentDynamicsMLP, NodeDeltaAttentionAutoEncoder
from course_project.utils import resolve_device

plt.rcParams['figure.dpi'] = 120


/home/alexz/Documents/course_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = random.SystemRandom().randrange(1, 2**31)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print('run seed:', seed)

cfg = {
    'dataset_path': '../data/new_reid_combined.pt',
    'analysis_export_dir': 'results/latent_space_simulator/analysis_exports',
    'cv_dataset': 'Reid CV2',
    'descriptor_output_dir': 'results/cvfest_reid_descriptor_projection',

    # Keep this True for the current Reid dataset so descriptor matches use the same systems.
    # Set False only when notebook-04 exports were regenerated from the same data file.
    'train_fresh_cvs': True,
    'training_latent_dims': [2],
    'train_count': 10,
    'val_count': 20,
    'split_seed': seed,
    'shuffle_dataset_within_source': True,
    'pos_dim': 2,
    'batch_graphs': 8,
    'early_stop_min_delta': 1e-5,
    'rollout_steps_grid': [100],
    'latent_tokens': 12,
    'hidden_size': 128,
    'edge_feature_dim': 16,
    'frame_stride': 2,
    'max_train_frames_per_sim': 20,
    'ae_max_epochs': 250,
    'ae_patience': 10,
    'ae_lr': 1e-3,
    'dyn_max_epochs': 250,
    'dyn_patience': 10,
    'dyn_lr': 1e-3,
    'device': 'auto',

    'match_final_p_ratio_tol': 1e-7,
    'ridge_alpha': 1e-2,
    'permutation_repeats': 64,
    'primary_projection_targets': ['final_p_ratio', 'mean_y_compression', 'final_y_compression'],
    'descriptor_focus_contains': ['stiffness', 'edge_len', 'hessian'],
    'random_seed': 0,
}


def resolve_existing_path(path_like):
    path = Path(path_like)
    if path.exists():
        return path
    if str(path).startswith('../'):
        repo_relative_path = Path(str(path)[3:])
        if repo_relative_path.exists():
            return repo_relative_path
    notebook_relative = Path('notebooks') / path
    if notebook_relative.exists():
        return notebook_relative
    return path

cfg['dataset_path'] = resolve_existing_path(cfg['dataset_path'])
cfg['analysis_export_dir'] = resolve_existing_path(cfg['analysis_export_dir'])
device = resolve_device(cfg['device'])
run_dir = Path(cfg['descriptor_output_dir'])
run_dir.mkdir(parents=True, exist_ok=True)
print('dataset:', cfg['dataset_path'])
print('exports:', cfg['analysis_export_dir'])
print('device:', device)
pd.DataFrame([cfg])


run seed: 996415490
dataset: ../data/new_reid_combined.pt
exports: results/latent_space_simulator/analysis_exports
device: cuda


,dataset_path,analysis_export_dir,cv_dataset,descriptor_output_dir,train_fresh_cvs,training_latent_dims,train_count,val_count,split_seed,shuffle_dataset_within_source,...,dyn_max_epochs,dyn_patience,dyn_lr,device,match_final_p_ratio_tol,ridge_alpha,permutation_repeats,primary_projection_targets,descriptor_focus_contains,random_seed
0,../data/new_reid_combined.pt,results/latent_space_simulator/analysis_exports,Reid CV2,results/cvfest_reid_descriptor_projection,True,[2],10,20,996415490,True,...,250,10,0.001,auto,1.000000e-07,0.01,64,"[final_p_ratio, mean_y_compression, final_y_co...","[stiffness, edge_len, hessian]",0


## Reid CV Training Regime

Notebook `07` now trains its own CV2 model on `new_reid_combined.pt` by default before descriptor projection. Set `cfg['train_fresh_cvs'] = False` only if notebook `04` has just been regenerated from the exact same Reid data file and you want to reuse those exports.


In [3]:
def make_frame_index(sims, *, stride=1, max_frames_per_sim=None, include_last=False):
    rows = []
    for sim_idx, sim in enumerate(sims):
        stop = len(sim) if include_last else len(sim) - 1
        frame_ids = list(range(0, stop, int(stride)))
        if max_frames_per_sim is not None and len(frame_ids) > int(max_frames_per_sim):
            pick = np.linspace(0, len(frame_ids) - 1, int(max_frames_per_sim)).round().astype(int)
            frame_ids = [frame_ids[i] for i in pick]
        rows.extend((sim_idx, int(t)) for t in frame_ids)
    return rows


def make_transition_index(sims, *, stride=1, max_frames_per_sim=None):
    rows = []
    for sim_idx, sim in enumerate(sims):
        frame_ids = list(range(0, len(sim) - 1, int(stride)))
        if max_frames_per_sim is not None and len(frame_ids) > int(max_frames_per_sim):
            pick = np.linspace(0, len(frame_ids) - 1, int(max_frames_per_sim)).round().astype(int)
            frame_ids = [frame_ids[i] for i in pick]
        rows.extend((sim_idx, int(t)) for t in frame_ids)
    return rows


def iter_batches(rows, batch_graphs, *, shuffle=True):
    rows = list(rows)
    if shuffle:
        random.shuffle(rows)
    for i in range(0, len(rows), int(batch_graphs)):
        yield rows[i:i + int(batch_graphs)]


def edge_features(ref_graph, cur_graph, *, device):
    ref_e = ref_graph.edge_attr.to(device).float()
    cur_e = cur_graph.edge_attr.to(device).float()
    ref_vec = ref_e[:, :cfg['pos_dim']]
    cur_vec = cur_e[:, :cfg['pos_dim']]
    ref_len = ref_e[:, cfg['pos_dim']:cfg['pos_dim'] + 1]
    cur_len = cur_e[:, cfg['pos_dim']:cfg['pos_dim'] + 1]
    stiffness = ref_e[:, -1:]
    stretch = cur_len - ref_len
    rel_stretch = stretch / ref_len.clamp_min(1e-6)
    return torch.cat([ref_vec, cur_vec, ref_len, cur_len, stretch, rel_stretch, stiffness, cur_e], dim=-1)


def frame_node_feature(sim, t, *, mode, device):
    t = int(t)
    cur_pos = sim[t].x[:, :cfg['pos_dim']].to(device).float()
    if mode in ('position', 'positions'):
        return cur_pos
    if mode == 'delta':
        ref_pos = sim[0].x[:, :cfg['pos_dim']].to(device).float()
        return cur_pos - ref_pos
    if mode == 'velocity':
        if t <= 0:
            return torch.zeros_like(cur_pos)
        prev_pos = sim[t - 1].x[:, :cfg['pos_dim']].to(device).float()
        return cur_pos - prev_pos
    raise ValueError(f'Unknown node_feature_mode: {mode}')


def batch_delta_graphs(sims, rows, *, device, node_feature_mode='delta'):
    xs = []
    node_features = []
    ref_xs = []
    edge_attrs = []
    ref_edge_attrs = []
    edge_indices = []
    batch = []
    node_offset = 0
    for local_idx, (sim_idx, t) in enumerate(rows):
        sim = sims[int(sim_idx)]
        ref_graph = sim[0]
        cur_graph = sim[int(t)]
        ref_pos = ref_graph.x[:, :cfg['pos_dim']].to(device).float()
        cur_pos = cur_graph.x[:, :cfg['pos_dim']].to(device).float()
        xs.append(cur_pos - ref_pos)
        node_features.append(frame_node_feature(sim, t, mode=node_feature_mode, device=device))
        ref_xs.append(ref_pos)
        edge_attrs.append(edge_features(ref_graph, cur_graph, device=device))
        ref_edge_attrs.append(edge_features(ref_graph, ref_graph, device=device))
        edge_indices.append(ref_graph.edge_index.to(device).long() + node_offset)
        batch.append(torch.full((ref_pos.size(0),), local_idx, dtype=torch.long, device=device))
        node_offset += ref_pos.size(0)
    return {
        'delta': torch.cat(xs, dim=0),
        'node_feature': torch.cat(node_features, dim=0),
        'ref_pos': torch.cat(ref_xs, dim=0),
        'edge_attr': torch.cat(edge_attrs, dim=0),
        'ref_edge_attr': torch.cat(ref_edge_attrs, dim=0),
        'edge_index': torch.cat(edge_indices, dim=1),
        'batch': torch.cat(batch, dim=0),
    }


def fit_delta_stats(sims, rows, *, device):
    chunks = []
    for rows_batch in iter_batches(rows, cfg['batch_graphs'], shuffle=False):
        batch_data = batch_delta_graphs(sims, rows_batch, device=device)
        chunks.append(batch_data['delta'].detach())
    all_delta = torch.cat(chunks, dim=0)
    mean = all_delta.mean(dim=0, keepdim=True)
    std = all_delta.std(dim=0, keepdim=True).clamp_min(1e-6)
    return mean, std


def fit_node_feature_stats(sims, rows, *, device, node_feature_mode):
    chunks = []
    for rows_batch in iter_batches(rows, cfg['batch_graphs'], shuffle=False):
        batch_data = batch_delta_graphs(sims, rows_batch, device=device, node_feature_mode=node_feature_mode)
        chunks.append(batch_data['node_feature'].detach())
    all_features = torch.cat(chunks, dim=0)
    mean = all_features.mean(dim=0, keepdim=True)
    std = all_features.std(dim=0, keepdim=True).clamp_min(1e-6)
    return mean, std


def fit_edge_stats(sims, rows, *, device):
    chunks = []
    for rows_batch in iter_batches(rows, cfg['batch_graphs'], shuffle=False):
        batch_data = batch_delta_graphs(sims, rows_batch, device=device)
        chunks.append(batch_data['edge_attr'].detach())
    all_edges = torch.cat(chunks, dim=0)
    mean = all_edges.mean(dim=0, keepdim=True)
    std = all_edges.std(dim=0, keepdim=True).clamp_min(1e-6)
    return mean, std


def normalize_delta(delta):
    return (delta - delta_mean) / delta_std


def unnormalize_delta(delta_norm):
    return delta_norm * delta_std + delta_mean


def normalize_node_feature(node_feature):
    return (node_feature - node_feature_mean) / node_feature_std


def normalize_edge_attr(edge_attr):
    return (edge_attr - edge_mean) / edge_std


def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - y_true.mean()) ** 2))
    return float('nan') if ss_tot <= 0 else 1.0 - ss_res / ss_tot


def pearson_r(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2 or np.allclose(y_true.std(), 0) or np.allclose(y_pred.std(), 0):
        return float('nan')
    return float(np.corrcoef(y_true, y_pred)[0, 1])


def encode_frame(model, sim, t, *, device):
    ref_graph = sim[0]
    cur_graph = sim[int(t)]
    ref_pos = ref_graph.x[:, :cfg['pos_dim']].to(device).float()
    node_feature = frame_node_feature(sim, t, mode=node_feature_mode, device=device)
    node_feature_norm = normalize_node_feature(node_feature)
    edge_attr_norm = normalize_edge_attr(edge_features(ref_graph, cur_graph, device=device))
    ref_edge_attr_norm = normalize_edge_attr(edge_features(ref_graph, ref_graph, device=device))
    edge_index = ref_graph.edge_index.to(device).long()
    batch = torch.zeros(ref_pos.size(0), dtype=torch.long, device=device)
    z, _ = model.encode(node_feature_norm, ref_pos, edge_attr_norm, ref_edge_attr_norm, edge_index, batch)
    return z.squeeze(0)


def encode_transition_batch(model, sims, rows, *, device):
    z0 = []
    z1 = []
    with torch.no_grad():
        for sim_idx, t in rows:
            sim = sims[int(sim_idx)]
            z0.append(encode_frame(model, sim, int(t), device=device))
            z1.append(encode_frame(model, sim, int(t) + 1, device=device))
    return torch.stack(z0, dim=0), torch.stack(z1, dim=0)


def fit_latent_step_stats(model, sims, rows, *, device):
    z_chunks = []
    dz_chunks = []
    for rows_batch in iter_batches(rows, cfg['batch_graphs'], shuffle=False):
        z0, z1 = encode_transition_batch(model, sims, rows_batch, device=device)
        z_chunks.append(z0.detach())
        dz_chunks.append((z1 - z0).detach())
    z_all = torch.cat(z_chunks, dim=0)
    dz_all = torch.cat(dz_chunks, dim=0)
    return {
        'z_mean': z_all.mean(dim=0, keepdim=True),
        'z_std': z_all.std(dim=0, keepdim=True).clamp_min(1e-6),
        'dz_mean': dz_all.mean(dim=0, keepdim=True),
        'dz_std': dz_all.std(dim=0, keepdim=True).clamp_min(1e-6),
    }


def normalize_z(z):
    return (z - latent_stats['z_mean']) / latent_stats['z_std']


def normalize_dz(dz):
    return (dz - latent_stats['dz_mean']) / latent_stats['dz_std']


def unnormalize_dz(dz_norm):
    return dz_norm * latent_stats['dz_std'] + latent_stats['dz_mean']


def latent_step(model, z):
    z_norm = normalize_z(z.unsqueeze(0))
    pred_dz_norm = model(z_norm) - z_norm
    return z + unnormalize_dz(pred_dz_norm).squeeze(0)


def decode_latent_to_graph(ae_model, sim, z, target_index, *, device):
    ref = clone_graph(sim[0]).to(device)
    ref_pos = ref.x[:, :cfg['pos_dim']].float()
    ref_edge_attr_norm = normalize_edge_attr(edge_features(sim[0], sim[0], device=device))
    batch = torch.zeros(ref_pos.size(0), dtype=torch.long, device=device)
    h0 = ae_model.encode_reference_graph(ref_pos, ref_edge_attr_norm, ref.edge_index.to(device).long())
    delta_norm = ae_model.decode(z.unsqueeze(0), h0, batch)
    delta = unnormalize_delta(delta_norm)
    pred = clone_graph(sim[target_index]).to(device)
    pred.x = pred.x.clone().float()
    pred.x[:, :cfg['pos_dim']] = ref_pos + delta
    return pred.cpu()


def _epoch_ae(model, sims, frame_rows, *, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    for rows in iter_batches(frame_rows, cfg['batch_graphs'], shuffle=is_train):
        batch_data = batch_delta_graphs(sims, rows, device=device, node_feature_mode=node_feature_mode)
        delta_norm = normalize_delta(batch_data['delta'])
        node_feature_norm = normalize_node_feature(batch_data['node_feature'])
        edge_attr_norm = normalize_edge_attr(batch_data['edge_attr'])
        ref_edge_attr_norm = normalize_edge_attr(batch_data['ref_edge_attr'])
        recon_norm, _ = model(node_feature_norm, batch_data['ref_pos'], edge_attr_norm, ref_edge_attr_norm, batch_data['edge_index'], batch_data['batch'])
        loss = F.mse_loss(recon_norm, delta_norm)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        losses.append(float(loss.item()))
    return float(np.mean(losses)) if losses else float('nan')


def _epoch_dyn(model, ae_model, sims, transition_rows, *, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    raw_losses = []
    for rows in iter_batches(transition_rows, cfg['batch_graphs'], shuffle=is_train):
        z0, z1 = encode_transition_batch(ae_model, sims, rows, device=device)
        z0_norm = normalize_z(z0)
        pred_dz_norm = model(z0_norm) - z0_norm
        target_dz_norm = normalize_dz(z1 - z0)
        loss = F.mse_loss(pred_dz_norm, target_dz_norm)
        if is_train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        losses.append(float(loss.item()))
        raw_losses.append(float(F.mse_loss(unnormalize_dz(pred_dz_norm), z1 - z0).item()))
    return {
        'loss_norm': float(np.mean(losses)) if losses else float('nan'),
        'loss_raw': float(np.mean(raw_losses)) if raw_losses else float('nan'),
    }


def p_ratio_metrics(df, *, dataset, split_name, rollout_steps):
    if len(df) == 0:
        return {'dataset': dataset, 'split': split_name, 'rollout_steps': rollout_steps, 'used': 0}
    return {
        'dataset': dataset,
        'split': split_name,
        'rollout_steps': int(rollout_steps),
        'used': len(df),
        'p_ratio_r2': r2_score(df['true_p_ratio'], df['pred_p_ratio']),
        'p_ratio_pearson': pearson_r(df['true_p_ratio'], df['pred_p_ratio']),
        'p_ratio_mse': float(np.mean((df['true_p_ratio'] - df['pred_p_ratio']) ** 2)),
        'final_pos_mse': float(df['final_pos_mse'].mean()),
        'initial_to_target_mse': float(df['initial_to_target_mse'].mean()),
        'pred_to_initial_mse': float(df['pred_to_initial_mse'].mean()),
        'movement_fraction_mse': float(df['movement_fraction_mse'].mean()),
    }


def latent_rollout_eval(ae_model, dyn_model, sims, *, dataset, split_name, rollout_steps):
    rows = []
    ae_model.eval(); dyn_model.eval()
    with torch.no_grad():
        for sim_idx, sim in enumerate(sims):
            target_index = min(int(rollout_steps), len(sim) - 1)
            if target_index <= 0:
                continue
            z0 = encode_frame(ae_model, sim, 0, device=device)
            z = z0.clone()
            for _ in range(target_index):
                z = latent_step(dyn_model, z)
            pred_graph = decode_latent_to_graph(ae_model, sim, z, target_index, device=device)
            pred_pr = float(calc_p_ratio_rollout_sides([clone_graph(sim[0]).cpu(), pred_graph], -1))
            true_pr = float(calc_p_ratio_rollout_sides(sim, target_index))
            initial_pos = sim[0].x[:, :cfg['pos_dim']].cpu().float()
            target_pos = sim[target_index].x[:, :cfg['pos_dim']].cpu().float()
            pred_pos = pred_graph.x[:, :cfg['pos_dim']].cpu().float()
            initial_to_target_mse = float(F.mse_loss(initial_pos, target_pos).item())
            pred_to_initial_mse = float(F.mse_loss(pred_pos, initial_pos).item())
            rows.append({
                'dataset': dataset,
                'split': split_name,
                'sim_idx': sim_idx,
                'target_index': target_index,
                'rollout_steps': int(rollout_steps),
                'pred_p_ratio': pred_pr,
                'true_p_ratio': true_pr,
                'final_pos_mse': float(F.mse_loss(pred_pos, target_pos).item()),
                'initial_to_target_mse': initial_to_target_mse,
                'pred_to_initial_mse': pred_to_initial_mse,
                'movement_fraction_mse': pred_to_initial_mse / initial_to_target_mse if initial_to_target_mse > 0 else float('nan'),
            })
    df = pd.DataFrame(rows)
    return df, p_ratio_metrics(df, dataset=dataset, split_name=split_name, rollout_steps=rollout_steps)


def latent_rollout_trajectory(ae_model, dyn_model, sim, *, rollout_steps, device):
    ae_model.eval(); dyn_model.eval()
    rows = []
    with torch.no_grad():
        z = encode_frame(ae_model, sim, 0, device=device)
        max_steps = min(int(rollout_steps), len(sim) - 1)
        for step_idx in range(max_steps + 1):
            true_pr = float(calc_p_ratio_rollout_sides(sim, step_idx))
            row = {
                'frame': int(step_idx),
                'true_p_ratio': true_pr,
            }
            z_np = z.detach().cpu().numpy().reshape(-1)
            for dim_idx, value in enumerate(z_np):
                row[f'z{dim_idx}'] = float(value)
            rows.append(row)
            if step_idx < max_steps:
                z = latent_step(dyn_model, z)
    return pd.DataFrame(rows)


def train_one_source(source_spec):
    global delta_mean, delta_std, node_feature_mean, node_feature_std, edge_mean, edge_std, latent_stats, node_feature_mode

    label = source_spec['label']
    p = {**cfg, **source_spec}
    print(f"\n=== {label} ===")
    train_data, val_data, test_data, split_info = resolve_dataset_splits(
        source_spec['path'],
        train_count=p['train_count'],
        val_count=p['val_count'],
        split_seed=cfg['split_seed'],
        shuffle_within_source=cfg.get('shuffle_dataset_within_source', False),
    )
    print(pd.DataFrame(split_info).to_string(index=False))

    node_feature_mode = p.get('node_feature_mode', 'delta')
    print('node_feature_mode:', node_feature_mode)
    train_frames = make_frame_index(train_data, stride=p['frame_stride'], max_frames_per_sim=p['max_train_frames_per_sim'], include_last=True)
    val_frames = make_frame_index(val_data, stride=max(1, p['frame_stride']), max_frames_per_sim=80, include_last=True)
    delta_mean, delta_std = fit_delta_stats(train_data, train_frames, device=device)
    node_feature_mean, node_feature_std = fit_node_feature_stats(train_data, train_frames, device=device, node_feature_mode=node_feature_mode)
    edge_mean, edge_std = fit_edge_stats(train_data, train_frames, device=device)
    p['edge_feature_dim'] = int(edge_mean.numel())
    print('edge_feature_dim:', p['edge_feature_dim'])

    ae_model = NodeDeltaAttentionAutoEncoder(
        pos_dim=cfg['pos_dim'], edge_dim=p['edge_feature_dim'], hidden_size=p['hidden_size'],
        latent_dim=p['latent_dim'], latent_tokens=p['latent_tokens'],
    ).to(device)
    ae_opt = torch.optim.AdamW(ae_model.parameters(), lr=p['ae_lr'], weight_decay=p.get('ae_weight_decay', 1e-5))
    best_ae_state = None
    best_val = float('inf')
    stale = 0
    ae_history = []
    print('autoencoder')
    for epoch in range(1, p['ae_max_epochs'] + 1):
        train_loss = _epoch_ae(ae_model, train_data, train_frames, optimizer=ae_opt)
        with torch.no_grad():
            val_loss = _epoch_ae(ae_model, val_data, val_frames, optimizer=None)
        ae_history.append({'dataset': label, 'latent_dim': p['latent_dim'], 'hidden_size': p['hidden_size'], 'epoch': epoch, 'train_mse_norm': train_loss, 'val_mse_norm': val_loss})
        if val_loss < best_val - cfg['early_stop_min_delta']:
            best_val = val_loss
            best_ae_state = {k: v.detach().cpu().clone() for k, v in ae_model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0 or stale == 0:
            print(f"ae {epoch:04d} train={train_loss:.6g} val={val_loss:.6g} stale={stale}")
        if stale >= p['ae_patience']:
            break
    ae_model.load_state_dict(best_ae_state)
    ae_model.eval()
    for param in ae_model.parameters():
        param.requires_grad_(False)

    train_steps = make_transition_index(train_data, stride=p['frame_stride'], max_frames_per_sim=p['max_train_frames_per_sim'])
    val_steps = make_transition_index(val_data, stride=max(1, p['frame_stride']), max_frames_per_sim=80)
    latent_stats = fit_latent_step_stats(ae_model, train_data, train_steps, device=device)
    dyn_model = LatentDynamicsMLP(p['latent_dim'], p['hidden_size']).to(device)
    dyn_opt = torch.optim.AdamW(dyn_model.parameters(), lr=p['dyn_lr'], weight_decay=p.get('dyn_weight_decay', 1e-5))
    best_dyn_state = None
    best_val = float('inf')
    stale = 0
    dyn_history = []
    print('latent dynamics')
    for epoch in range(1, p['dyn_max_epochs'] + 1):
        train_info = _epoch_dyn(dyn_model, ae_model, train_data, train_steps, optimizer=dyn_opt)
        with torch.no_grad():
            val_info = _epoch_dyn(dyn_model, ae_model, val_data, val_steps, optimizer=None)
        dyn_history.append({'dataset': label, 'latent_dim': p['latent_dim'], 'hidden_size': p['hidden_size'], 'epoch': epoch, 'train_dz_mse_norm': train_info['loss_norm'], 'val_dz_mse_norm': val_info['loss_norm'], 'val_dz_mse_raw': val_info['loss_raw']})
        if val_info['loss_norm'] < best_val - cfg['early_stop_min_delta']:
            best_val = val_info['loss_norm']
            best_dyn_state = {k: v.detach().cpu().clone() for k, v in dyn_model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0 or stale == 0:
            print(f"dyn {epoch:04d} train={train_info['loss_norm']:.6g} val={val_info['loss_norm']:.6g} stale={stale}")
        if stale >= p['dyn_patience']:
            break
    dyn_model.load_state_dict(best_dyn_state)
    dyn_model.eval()

    rollout_rows = []
    rollout_stats = []
    for steps in cfg['rollout_steps_grid']:
        for split_name, sims in [('val', val_data), ('test', test_data)]:
            df, stats = latent_rollout_eval(ae_model, dyn_model, sims, dataset=label, split_name=split_name, rollout_steps=steps)
            rollout_rows.append(df)
            rollout_stats.append(stats)

    return {
        'label': label,
        'params': p,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data,
        'ae': ae_model,
        'dyn': dyn_model,
        'stats': {k: v.detach().cpu() for k, v in {'delta_mean': delta_mean, 'delta_std': delta_std, 'node_feature_mean': node_feature_mean, 'node_feature_std': node_feature_std, 'edge_mean': edge_mean, 'edge_std': edge_std, **latent_stats}.items()},
        'ae_history': pd.DataFrame(ae_history),
        'dyn_history': pd.DataFrame(dyn_history),
        'rollout_rows': pd.concat(rollout_rows, ignore_index=True),
        'rollout_stats': pd.DataFrame(rollout_stats),
    }


In [4]:
def reid_spec(latent_dim):
    return {
        'label': f'Reid CV{latent_dim}',
        'path': str(cfg['dataset_path']),
        'train_count': cfg['train_count'],
        'val_count': cfg['val_count'],
        'latent_dim': int(latent_dim),
        'latent_tokens': cfg['latent_tokens'],
        'hidden_size': cfg['hidden_size'],
        'edge_feature_dim': cfg['edge_feature_dim'],
        'frame_stride': cfg['frame_stride'],
        'max_train_frames_per_sim': cfg['max_train_frames_per_sim'],
        'ae_max_epochs': cfg['ae_max_epochs'],
        'ae_patience': cfg['ae_patience'],
        'ae_lr': cfg['ae_lr'],
        'dyn_max_epochs': cfg['dyn_max_epochs'],
        'dyn_patience': cfg['dyn_patience'],
        'dyn_lr': cfg['dyn_lr'],
    }


source_results = []
if cfg['train_fresh_cvs']:
    cfg['datasets'] = [reid_spec(k) for k in cfg['training_latent_dims']]
    source_results = [train_one_source(spec) for spec in cfg['datasets']]
    ae_history_df = pd.concat([r['ae_history'] for r in source_results], ignore_index=True)
    dyn_history_df = pd.concat([r['dyn_history'] for r in source_results], ignore_index=True)
    rollout_rows_df = pd.concat([r['rollout_rows'] for r in source_results], ignore_index=True)
    rollout_stats_df = pd.concat([r['rollout_stats'] for r in source_results], ignore_index=True)
    display(rollout_stats_df.round(4))
else:
    print('Skipping in-notebook CV training; using notebook-04 exports. Set cfg["train_fresh_cvs"] = True to train here.')



=== Reid CV2 ===
           source                         path  total  train  val  test
new_reid_combined ../data/new_reid_combined.pt    102     10   20    72
node_feature_mode: delta
edge_feature_dim: 13
autoencoder
ae 0001 train=0.718547 val=0.565732 stale=0
ae 0002 train=0.551293 val=0.473057 stale=0
ae 0004 train=0.491506 val=0.460842 stale=0
ae 0005 train=0.470939 val=0.450782 stale=0
ae 0006 train=0.398474 val=0.435473 stale=0
ae 0007 train=0.334167 val=0.369067 stale=0
ae 0008 train=0.295922 val=0.348665 stale=0
ae 0010 train=0.273086 val=0.367023 stale=2
ae 0015 train=0.24014 val=0.346346 stale=0
ae 0016 train=0.226818 val=0.342002 stale=0
ae 0020 train=0.204148 val=0.350802 stale=4
latent dynamics
dyn 0001 train=0.474165 val=0.187732 stale=0
dyn 0002 train=0.1257 val=0.12085 stale=0
dyn 0010 train=0.0798815 val=0.156012 stale=8


,dataset,split,rollout_steps,used,p_ratio_r2,p_ratio_pearson,p_ratio_mse,final_pos_mse,initial_to_target_mse,pred_to_initial_mse,movement_fraction_mse
0,Reid CV2,val,100,20,0.7109,0.8636,0.0117,0.0,0.0,0.0001,1.6114
1,Reid CV2,test,100,72,0.5468,0.8425,0.0140,0.0,0.0,0.0001,1.6419


## Load Discovered CVs

With `cfg['train_fresh_cvs'] = True`, this section uses the fresh Reid latent coordinates. If export reuse is enabled, it expects notebook `04` exports for the same Reid label and prefers `axis_p_ratio` before falling back to raw latent coordinates.


In [ ]:
export_dir = Path(cfg['analysis_export_dir'])


def read_export(name):
    path = export_dir / f'{name}.csv'
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def use_result_stats(result):
    stats = result['stats']
    global delta_mean, delta_std, node_feature_mean, node_feature_std, edge_mean, edge_std, latent_stats, node_feature_mode
    delta_mean = stats['delta_mean'].to(device)
    delta_std = stats['delta_std'].to(device)
    node_feature_mode = result['params'].get('node_feature_mode', 'delta')
    node_feature_mean = stats.get('node_feature_mean', stats['delta_mean']).to(device)
    node_feature_std = stats.get('node_feature_std', stats['delta_std']).to(device)
    edge_mean = stats['edge_mean'].to(device)
    edge_std = stats['edge_std'].to(device)
    latent_stats = {
        'z_mean': stats['z_mean'].to(device),
        'z_std': stats['z_std'].to(device),
        'dz_mean': stats['dz_mean'].to(device),
        'dz_std': stats['dz_std'].to(device),
    }


def latent_frame_table(result, sims, *, split):
    use_result_stats(result)
    rows = []
    with torch.no_grad():
        for sim_idx, sim in enumerate(sims):
            frame_count = len(sim)
            final_p_ratio = float(calc_p_ratio_rollout_sides(sim, -1))
            for frame_idx in range(frame_count):
                z = encode_frame(result['ae'], sim, frame_idx, device=device).detach().cpu().numpy().reshape(-1)
                row = {
                    'dataset': result['label'],
                    'latent_dim': int(result['params']['latent_dim']),
                    'split': split,
                    'sim_idx': int(sim_idx),
                    'frame': int(frame_idx),
                    'trajectory_progress': float(frame_idx / max(frame_count - 1, 1)),
                    'p_ratio': float(calc_p_ratio_rollout_sides(sim, frame_idx)),
                    'final_p_ratio': final_p_ratio,
                }
                for dim_idx, value in enumerate(z):
                    row[f'z{dim_idx}'] = float(value)
                rows.append(row)
    return pd.DataFrame(rows)


def span_strain(ref_graph, graph, axis):
    ref_pos = ref_graph.x[:, :cfg['pos_dim']].detach().cpu().float()
    cur_pos = graph.x[:, :cfg['pos_dim']].detach().cpu().float()
    ref_span = float(ref_pos[:, axis].max() - ref_pos[:, axis].min())
    cur_span = float(cur_pos[:, axis].max() - cur_pos[:, axis].min())
    if abs(ref_span) <= 1e-12:
        return float('nan')
    return (cur_span - ref_span) / abs(ref_span)


def physical_property_table(result, sims, *, split):
    rows = []
    for sim_idx, sim in enumerate(sims):
        ref = clone_graph(sim[0]).cpu()
        for frame_idx, graph in enumerate(sim):
            cur = clone_graph(graph).cpu()
            vertical_strain = span_strain(ref, cur, axis=1)
            rows.append({
                'dataset': result['label'],
                'latent_dim': int(result['params']['latent_dim']),
                'split': split,
                'sim_idx': int(sim_idx),
                'frame': int(frame_idx),
                'vertical_strain': vertical_strain,
                'y_compression': -vertical_strain,
            })
    return pd.DataFrame(rows)


if cfg['train_fresh_cvs']:
    if not source_results:
        raise RuntimeError('cfg["train_fresh_cvs"] is True, but source_results is empty. Run the optional training cell above.')
    result = next((r for r in source_results if r['label'] == cfg['cv_dataset']), source_results[0])
    cfg['cv_dataset'] = result['label']
    latent_frame_df = pd.concat([
        latent_frame_table(result, result['train_data'], split='train'),
        latent_frame_table(result, result['val_data'], split='val'),
        latent_frame_table(result, result['test_data'], split='test'),
    ], ignore_index=True)
    physical_property_df = pd.concat([
        physical_property_table(result, result['train_data'], split='train'),
        physical_property_table(result, result['val_data'], split='val'),
        physical_property_table(result, result['test_data'], split='test'),
    ], ignore_index=True)
    cv_frame_df = latent_frame_df.merge(
        physical_property_df,
        on=['dataset', 'latent_dim', 'split', 'sim_idx', 'frame'],
        how='left',
    )
    z_cols = [col for col in cv_frame_df.columns if col.startswith('z') and cv_frame_df[col].notna().any()]
    cv_signal_cols = z_cols
    cv_source = 'fresh_training_in_07'
    disentangled_scores_df = cv_frame_df.copy()
    disentangled_axes_df = pd.DataFrame()
else:
    latent_frame_df = read_export('latent_frame_rows')
    disentangled_scores_df = read_export('disentangled_scores')
    disentangled_axes_df = read_export('disentangled_axes')
    physical_property_df = read_export('physical_property_rows')
    if not physical_property_df.empty and 'vertical_strain' in physical_property_df.columns:
        physical_property_df['y_compression'] = -physical_property_df['vertical_strain']

    if not disentangled_scores_df.empty and cfg['cv_dataset'] in set(disentangled_scores_df['dataset']):
        cv_frame_df = disentangled_scores_df[disentangled_scores_df['dataset'].eq(cfg['cv_dataset'])].copy()
        cv_frame_df['frame_frac'] = cv_frame_df.get('trajectory_progress', np.nan)
        cv_signal_cols = [col for col in ['axis_p_ratio'] if col in cv_frame_df.columns]
        cv_source = 'disentangled_scores_export'
    else:
        cv_frame_df = latent_frame_df[latent_frame_df['dataset'].eq(cfg['cv_dataset'])].copy()
        if 'trajectory_progress' not in cv_frame_df.columns and 'frame_frac' in cv_frame_df.columns:
            cv_frame_df['trajectory_progress'] = cv_frame_df['frame_frac']
        cv_signal_cols = [col for col in cv_frame_df.columns if col.startswith('z') and cv_frame_df[col].notna().any()]
        cv_source = 'latent_frame_rows_export'

    if not physical_property_df.empty:
        phys_cols = ['dataset', 'split', 'sim_idx', 'frame', 'vertical_strain', 'y_compression']
        phys_cols = [col for col in phys_cols if col in physical_property_df.columns]
        cv_frame_df = cv_frame_df.merge(
            physical_property_df[phys_cols].drop_duplicates(['dataset', 'split', 'sim_idx', 'frame']),
            on=['dataset', 'split', 'sim_idx', 'frame'],
            how='left',
        )

if cv_frame_df.empty:
    raise FileNotFoundError(f'No CV frame data found for {cfg["cv_dataset"]} in {export_dir}')

print('CV source:', cv_source)
print('CV dataset:', cfg['cv_dataset'])
print('CV signals kept:', cv_signal_cols)
print('rows:', len(cv_frame_df), 'trajectories:', cv_frame_df['sim_idx'].nunique())
display(cv_frame_df.head())
if not disentangled_axes_df.empty:
    display(disentangled_axes_df[disentangled_axes_df['dataset'].eq(cfg['cv_dataset'])].round(4))


## Trajectory CV Summaries

Collapse each trajectory into interpretable CV summaries: mean, final, initial, slope, range, and early-time slope. These are the CVFEST projection targets.

In [ ]:
def safe_corr(a, b):
    a = pd.Series(a, dtype='float64')
    b = pd.Series(b, dtype='float64')
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3:
        return np.nan
    av = a[mask]
    bv = b[mask]
    if np.isclose(av.std(), 0.0) or np.isclose(bv.std(), 0.0):
        return np.nan
    return float(av.corr(bv))


def r2_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 3:
        return np.nan
    y = y_true[mask]
    p = y_pred[mask]
    ss_res = float(np.sum((y - p) ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    return np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot


summary_rows = []
for (dataset, split, sim_idx), group in cv_frame_df.groupby(['dataset', 'split', 'sim_idx'], sort=False):
    group = group.sort_values('frame')
    row = {
        'dataset': dataset,
        'split': split,
        'sim_idx': int(sim_idx),
        'frames': int(len(group)),
        'final_p_ratio': float(group['final_p_ratio'].dropna().iloc[0]) if 'final_p_ratio' in group and group['final_p_ratio'].notna().any() else np.nan,
    }
    if 'y_compression' in group:
        row['mean_y_compression'] = float(group['y_compression'].mean())
        row['final_y_compression'] = float(group['y_compression'].iloc[-1])
    if 'axis_p_ratio' in group:
        row['mean_axis_p_ratio'] = float(group['axis_p_ratio'].mean())
        row['final_axis_p_ratio'] = float(group['axis_p_ratio'].iloc[-1])
    summary_rows.append(row)

trajectory_cv_summary_df = pd.DataFrame(summary_rows)
target_cols = [col for col in cfg['primary_projection_targets'] if col in trajectory_cv_summary_df.columns]
print('projection targets:', target_cols)
display(trajectory_cv_summary_df.head().round(4))


## Recover Original Simulation Indices

Notebook 04 exports local split indices. To attach initial-graph descriptors, match each exported trajectory back to the raw dataset by final p-ratio. If matches are not unique, inspect the warning table before trusting the projection.

In [ ]:
all_sims = load_dataset(cfg['dataset_path'])
raw_rows = []
for original_idx, sim in enumerate(all_sims):
    raw_rows.append({
        'original_sim_idx': int(original_idx),
        'raw_frames': int(len(sim)),
        'raw_final_p_ratio': float(calc_p_ratio_rollout_sides(sim, -1)),
    })
raw_final_df = pd.DataFrame(raw_rows)

candidate_pairs = []
for summary_row_idx, row in trajectory_cv_summary_df.reset_index(drop=True).iterrows():
    final_p_ratio = float(row['final_p_ratio'])
    diffs = (raw_final_df['raw_final_p_ratio'] - final_p_ratio).abs()
    for original_idx, diff in zip(raw_final_df['original_sim_idx'], diffs):
        candidate_pairs.append((float(diff), int(summary_row_idx), int(original_idx)))

assigned_summary = set()
assigned_original = set()
assignments = {}
for diff, summary_row_idx, original_idx in sorted(candidate_pairs, key=lambda item: item[0]):
    if summary_row_idx in assigned_summary or original_idx in assigned_original:
        continue
    assigned_summary.add(summary_row_idx)
    assigned_original.add(original_idx)
    assignments[summary_row_idx] = (original_idx, diff)
    if len(assigned_summary) == len(trajectory_cv_summary_df):
        break

match_rows = []
summary_reset = trajectory_cv_summary_df.reset_index(drop=True)
for summary_row_idx, row in summary_reset.iterrows():
    if summary_row_idx not in assignments:
        raise RuntimeError(f'Could not assign an original Reid trajectory for summary row {summary_row_idx}.')
    original_idx, diff = assignments[summary_row_idx]
    raw_row = raw_final_df[raw_final_df['original_sim_idx'].eq(original_idx)].iloc[0]
    match_rows.append({
        'dataset': row['dataset'],
        'split': row['split'],
        'sim_idx': int(row['sim_idx']),
        'final_p_ratio': float(row['final_p_ratio']),
        'original_sim_idx': int(original_idx),
        'raw_final_p_ratio': float(raw_row['raw_final_p_ratio']),
        'abs_final_p_ratio_diff': float(diff),
        'raw_frames': int(raw_row['raw_frames']),
    })

sim_index_map_df = pd.DataFrame(match_rows)
max_diff = sim_index_map_df['abs_final_p_ratio_diff'].max()
print('matched trajectories:', len(sim_index_map_df), 'max final-p-ratio diff:', max_diff)
if max_diff > cfg['match_final_p_ratio_tol']:
    warnings.warn('Some final-p-ratio matches exceed tolerance. Check sim_index_map_df before interpreting descriptor projections.')
display(sim_index_map_df.sort_values('abs_final_p_ratio_diff', ascending=False).head(10).round(10))
trajectory_cv_summary_df = trajectory_cv_summary_df.merge(
    sim_index_map_df[['dataset', 'split', 'sim_idx', 'original_sim_idx', 'abs_final_p_ratio_diff', 'raw_frames']],
    on=['dataset', 'split', 'sim_idx'],
    how='left',
)


## Initial-System Descriptors

Compute descriptors available before rollout: geometry, topology, stiffness, and Hessian soft-mode spectrum from the initial graph.

In [ ]:
def unique_edge_pairs(edge_index):
    pairs = set()
    edge_index_np = edge_index.detach().cpu().numpy()
    for e in range(edge_index_np.shape[1]):
        i, j = int(edge_index_np[0, e]), int(edge_index_np[1, e])
        if i == j:
            continue
        pairs.add((i, j) if i < j else (j, i))
    return sorted(pairs)


def descriptor_row_for_initial_graph(sim, original_sim_idx):
    graph = sim[0]
    pos = graph.x[:, :2].detach().cpu().float().numpy()
    edge_index = graph.edge_index.detach().cpu()
    edge_attr = graph.edge_attr.detach().cpu().float().numpy()
    pairs = unique_edge_pairs(edge_index)
    n = int(pos.shape[0])
    degree = np.zeros(n, dtype=float)
    edge_lengths = []
    stiffness = []
    edge_index_np = edge_index.numpy()
    pair_to_stiffness = {}
    for e in range(edge_index_np.shape[1]):
        i, j = int(edge_index_np[0, e]), int(edge_index_np[1, e])
        if i == j:
            continue
        key = (i, j) if i < j else (j, i)
        if key not in pair_to_stiffness:
            pair_to_stiffness[key] = float(edge_attr[e, -1]) if edge_attr.size else np.nan
    for i, j in pairs:
        degree[i] += 1.0
        degree[j] += 1.0
        edge_lengths.append(float(np.linalg.norm(pos[i] - pos[j])))
        stiffness.append(pair_to_stiffness.get((i, j), np.nan))
    edge_lengths = np.asarray(edge_lengths, dtype=float)
    stiffness = np.asarray(stiffness, dtype=float)
    x = pos[:, 0]
    y = pos[:, 1]
    x_span = float(x.max() - x.min())
    y_span = float(y.max() - y.min())
    x_mid = 0.5 * (x.max() + x.min())
    y_mid = 0.5 * (y.max() + y.min())
    left = x <= np.quantile(x, 0.15)
    right = x >= np.quantile(x, 0.85)
    bottom = y <= np.quantile(y, 0.15)
    top = y >= np.quantile(y, 0.85)
    row = {
        'original_sim_idx': int(original_sim_idx),
        'node_count': n,
        'edge_count': int(len(pairs)),
        'edge_density': float(len(pairs) / max(n * (n - 1) / 2, 1)),
        'mean_degree': float(np.mean(degree)),
        'std_degree': float(np.std(degree)),
        'max_degree': float(np.max(degree)),
        'min_degree': float(np.min(degree)),
        'x_span0': x_span,
        'y_span0': y_span,
        'aspect_ratio0': x_span / y_span if abs(y_span) > 1e-12 else np.nan,
        'x_center0': float(x_mid),
        'y_center0': float(y_mid),
        'edge_len_mean': float(np.nanmean(edge_lengths)),
        'edge_len_std': float(np.nanstd(edge_lengths)),
        'edge_len_min': float(np.nanmin(edge_lengths)),
        'edge_len_max': float(np.nanmax(edge_lengths)),
        'stiffness_mean': float(np.nanmean(stiffness)),
        'stiffness_std': float(np.nanstd(stiffness)),
        'stiffness_min': float(np.nanmin(stiffness)),
        'stiffness_max': float(np.nanmax(stiffness)),
        'left_mean_degree': float(np.mean(degree[left])),
        'right_mean_degree': float(np.mean(degree[right])),
        'top_mean_degree': float(np.mean(degree[top])),
        'bottom_mean_degree': float(np.mean(degree[bottom])),
        'left_right_degree_asymmetry': float(np.mean(degree[right]) - np.mean(degree[left])),
        'top_bottom_degree_asymmetry': float(np.mean(degree[top]) - np.mean(degree[bottom])),
    }
    try:
        r0 = make_r0_dict(graph, pos_dim=2)
        H = build_spring_hessian_2d(graph, r0_dict=r0, pos_dim=2)
        evals, evecs = np.linalg.eigh(H)
        order = np.argsort(evals)
        evals = evals[order]
        evecs = evecs[:, order]
        for idx in range(min(12, len(evals))):
            row[f'hessian_lambda{idx + 1}'] = float(evals[idx])
        for mode_idx in [4, 5]:
            if evecs.shape[1] > mode_idx:
                vec = evecs[:, mode_idx].reshape(n, 2)
                node_amp2 = np.sum(vec ** 2, axis=1)
                denom = n * float(np.sum(node_amp2 ** 2))
                row[f'hessian_mode{mode_idx + 1}_participation'] = float((np.sum(node_amp2) ** 2) / denom) if denom > 1e-12 else np.nan
                row[f'hessian_mode{mode_idx + 1}_x_fraction'] = float(np.sum(vec[:, 0] ** 2) / np.sum(vec ** 2)) if np.sum(vec ** 2) > 1e-12 else np.nan
        row['hessian_gap_6_5'] = row.get('hessian_lambda6', np.nan) - row.get('hessian_lambda5', np.nan)
    except Exception as exc:
        warnings.warn(f'Hessian descriptors failed for sim {original_sim_idx}: {exc}')
    return row

needed_original_indices = sorted(trajectory_cv_summary_df['original_sim_idx'].dropna().astype(int).unique())
descriptor_rows = [descriptor_row_for_initial_graph(all_sims[idx], idx) for idx in needed_original_indices]
initial_descriptor_df = pd.DataFrame(descriptor_rows)
print('descriptor rows:', len(initial_descriptor_df), 'columns:', len(initial_descriptor_df.columns))
display(initial_descriptor_df.head().round(4))


## Project CVs Onto Descriptors

First inspect simple correlations; then fit a standardized ridge projection and use permutation-style importance to identify descriptors that matter jointly.

In [ ]:
cv_descriptor_df = trajectory_cv_summary_df.merge(initial_descriptor_df, on='original_sim_idx', how='left')

descriptor_cols = [
    col for col in initial_descriptor_df.columns
    if col != 'original_sim_idx' and pd.api.types.is_numeric_dtype(initial_descriptor_df[col])
]
if cfg.get('descriptor_focus_contains'):
    keep_parts = tuple(cfg['descriptor_focus_contains'])
    descriptor_cols = [col for col in descriptor_cols if any(part in col for part in keep_parts)]
# Drop constants.
descriptor_cols = [col for col in descriptor_cols if cv_descriptor_df[col].nunique(dropna=True) > 1]

corr_rows = []
for target in target_cols:
    for desc in descriptor_cols:
        r = safe_corr(cv_descriptor_df[desc], cv_descriptor_df[target])
        corr_rows.append({'target': target, 'descriptor': desc, 'pearson_r': r, 'abs_r': abs(r) if np.isfinite(r) else np.nan})
cv_descriptor_corr_df = pd.DataFrame(corr_rows)

top_corr_df = (
    cv_descriptor_corr_df
    .sort_values('abs_r', ascending=False)
    .groupby('target', as_index=False)
    .head(8)
    .drop(columns='abs_r')
)
display(top_corr_df.round(4))


In [ ]:
def standardize_matrix(df, cols):
    X = df[cols].to_numpy(dtype=float)
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd = np.where(sd <= 1e-12, 1.0, sd)
    Xs = (X - mu) / sd
    Xs = np.where(np.isfinite(Xs), Xs, 0.0)
    return Xs, mu, sd


def fit_ridge(X, y, alpha):
    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    Xv = X[mask]
    yv = y[mask]
    if len(yv) < 5:
        return None, mask
    X_aug = np.column_stack([Xv, np.ones(len(Xv))])
    reg = np.eye(X_aug.shape[1]) * float(alpha)
    reg[-1, -1] = 0.0
    coef = np.linalg.solve(X_aug.T @ X_aug + reg, X_aug.T @ yv)
    return coef, mask


def predict_ridge(X, coef):
    return np.column_stack([X, np.ones(X.shape[0])]) @ coef


rng = np.random.default_rng(int(cfg['random_seed']))
X_desc, _, _ = standardize_matrix(cv_descriptor_df, descriptor_cols)
projection_rows = []
importance_rows = []
for target in target_cols:
    y_raw = cv_descriptor_df[target].to_numpy(dtype=float)
    y_sd = np.nanstd(y_raw)
    y = (y_raw - np.nanmean(y_raw)) / (y_sd if y_sd > 1e-12 else 1.0)
    coef, mask = fit_ridge(X_desc, y, alpha=cfg['ridge_alpha'])
    if coef is None:
        continue
    pred = predict_ridge(X_desc[mask], coef)
    base_mse = float(np.mean((y[mask] - pred) ** 2))
    target_r2 = r2_score(y[mask], pred)
    for col_idx, (desc, weight) in enumerate(zip(descriptor_cols, coef[:-1])):
        projection_rows.append({
            'target': target,
            'descriptor': desc,
            'standardized_weight': float(weight),
            'target_r2': target_r2,
        })
        increases = []
        for _ in range(int(cfg['permutation_repeats'])):
            X_perm = X_desc[mask].copy()
            X_perm[:, col_idx] = rng.permutation(X_perm[:, col_idx])
            perm_pred = predict_ridge(X_perm, coef)
            increases.append(float(np.mean((y[mask] - perm_pred) ** 2)) - base_mse)
        importance_rows.append({
            'target': target,
            'descriptor': desc,
            'mse_increase_when_permuted': float(np.mean(increases)),
            'mse_increase_std': float(np.std(increases)),
            'target_r2': target_r2,
        })

cv_descriptor_projection_df = pd.DataFrame(projection_rows)
cv_descriptor_importance_df = pd.DataFrame(importance_rows)

display(
    cv_descriptor_projection_df
    .assign(abs_weight=lambda df: df['standardized_weight'].abs())
    .sort_values('abs_weight', ascending=False)
    .groupby('target', as_index=False)
    .head(8)
    .drop(columns='abs_weight')
    .round(4)
)

display(
    cv_descriptor_importance_df
    .sort_values('mse_increase_when_permuted', ascending=False)
    .groupby('target', as_index=False)
    .head(8)
    .round(4)
)


## Stiffness Intervention Screen

Use the fitted descriptor model as a cheap intervention screen. This does not rerun dynamics; it predicts how simple stiffness edits would move final p-ratio and y-compression according to the descriptor projection.


In [ ]:
def clone_sim_with_modified_initial_stiffness(sim, modifier):
    sim_mod = list(sim)
    g0 = clone_graph(sim[0])
    g0.edge_attr = g0.edge_attr.clone().float()
    k = g0.edge_attr[:, -1].detach().cpu().numpy().astype(float)
    k_new = np.asarray(modifier(k), dtype=float)
    k_new = np.clip(k_new, 1e-8, None)
    g0.edge_attr[:, -1] = torch.as_tensor(k_new, dtype=g0.edge_attr.dtype, device=g0.edge_attr.device)
    sim_mod[0] = g0
    return sim_mod


def predict_targets_from_descriptors(descriptor_df, targets):
    pred_rows = []
    for target in targets:
        X_train, x_mu, x_sd = standardize_matrix(cv_descriptor_df, descriptor_cols)
        y_raw = cv_descriptor_df[target].to_numpy(dtype=float)
        y_mu = float(np.nanmean(y_raw))
        y_sd = float(np.nanstd(y_raw)) if np.nanstd(y_raw) > 1e-12 else 1.0
        y = (y_raw - y_mu) / y_sd
        coef, mask = fit_ridge(X_train, y, alpha=cfg['ridge_alpha'])
        if coef is None:
            continue
        X = descriptor_df[descriptor_cols].to_numpy(dtype=float)
        Xs = (X - x_mu) / x_sd
        Xs = np.where(np.isfinite(Xs), Xs, 0.0)
        pred = predict_ridge(Xs, coef) * y_sd + y_mu
        for idx, value in zip(descriptor_df.index, pred):
            pred_rows.append({'row_index': idx, 'target': target, 'predicted_value': float(value)})
    return pd.DataFrame(pred_rows)


intervention_specs = [
    ('baseline', lambda k: k),
    ('all_stiffness_plus_10pct', lambda k: k * 1.10),
    ('all_stiffness_minus_10pct', lambda k: k * 0.90),
    ('weak_springs_floor_to_q25', lambda k: np.maximum(k, np.quantile(k, 0.25))),
    ('weak_springs_plus_25pct', lambda k: np.where(k <= np.quantile(k, 0.25), k * 1.25, k)),
    ('half_homogenize_to_mean', lambda k: 0.5 * k + 0.5 * np.mean(k)),
]

intervention_descriptor_rows = []
for original_idx in sorted(cv_descriptor_df['original_sim_idx'].dropna().astype(int).unique()):
    sim = all_sims[original_idx]
    for intervention, modifier in intervention_specs:
        sim_mod = clone_sim_with_modified_initial_stiffness(sim, modifier)
        row = descriptor_row_for_initial_graph(sim_mod, original_idx)
        row['intervention'] = intervention
        intervention_descriptor_rows.append(row)
intervention_descriptor_df = pd.DataFrame(intervention_descriptor_rows)

intervention_pred_long_df = predict_targets_from_descriptors(intervention_descriptor_df, target_cols)
intervention_pred_df = intervention_pred_long_df.pivot(index='row_index', columns='target', values='predicted_value')
intervention_screen_df = pd.concat([intervention_descriptor_df.reset_index(drop=True), intervention_pred_df.reset_index(drop=True)], axis=1)

baseline = intervention_screen_df[intervention_screen_df['intervention'].eq('baseline')].set_index('original_sim_idx')
shift_rows = []
for _, row in intervention_screen_df.iterrows():
    base = baseline.loc[row['original_sim_idx']]
    for target in target_cols:
        shift_rows.append({
            'original_sim_idx': int(row['original_sim_idx']),
            'intervention': row['intervention'],
            'target': target,
            'predicted_value': float(row[target]),
            'predicted_shift_from_baseline': float(row[target] - base[target]),
        })
intervention_shift_df = pd.DataFrame(shift_rows)
intervention_summary_df = (
    intervention_shift_df[~intervention_shift_df['intervention'].eq('baseline')]
    .groupby(['intervention', 'target'], as_index=False)
    .agg(mean_shift=('predicted_shift_from_baseline', 'mean'), median_shift=('predicted_shift_from_baseline', 'median'), abs_mean_shift=('predicted_shift_from_baseline', lambda x: float(np.mean(np.abs(x)))))
    .sort_values(['target', 'mean_shift'], ascending=[True, False])
)
display(intervention_summary_df.round(4))


## Visual Summary

In [ ]:
main_targets = target_cols

top_descriptors = (
    cv_descriptor_corr_df[cv_descriptor_corr_df['target'].isin(main_targets)]
    .groupby('descriptor')['abs_r']
    .max()
    .sort_values(ascending=False)
    .head(10)
    .index.tolist()
)
heat = (
    cv_descriptor_corr_df[cv_descriptor_corr_df['target'].isin(main_targets) & cv_descriptor_corr_df['descriptor'].isin(top_descriptors)]
    .pivot(index='target', columns='descriptor', values='pearson_r')
    .reindex(index=main_targets, columns=top_descriptors)
)
fig, axes = plt.subplots(1, 2, figsize=(17, 5.2), constrained_layout=True)
ax = axes[0]
im = ax.imshow(heat.to_numpy(dtype=float), vmin=-1, vmax=1, cmap='coolwarm', aspect='auto')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=35, ha='right')
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title('Targets vs initial stiffness/edge/Hessian descriptors')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        if np.isfinite(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', color='white' if abs(val) > 0.55 else 'black', fontsize=8)
fig.colorbar(im, ax=ax, label='Pearson r')

if 'intervention_summary_df' in globals() and not intervention_summary_df.empty:
    plot_target = 'final_p_ratio' if 'final_p_ratio' in set(intervention_summary_df['target']) else intervention_summary_df['target'].iloc[0]
    work = intervention_summary_df[intervention_summary_df['target'].eq(plot_target)].sort_values('mean_shift')
    axes[1].barh(work['intervention'], work['mean_shift'], color=np.where(work['mean_shift'] >= 0, '#4C78A8', '#F58518'))
    axes[1].axvline(0, color='black', lw=1)
    axes[1].set_xlabel(f'predicted mean shift in {plot_target}')
    axes[1].set_title('Stiffness intervention screen')
    axes[1].grid(axis='x', alpha=0.25)
else:
    axes[1].axis('off')
plt.show()

# Pair plots for final p-ratio and y-compression targets.
plot_pairs = []
for target in main_targets:
    row = cv_descriptor_corr_df[cv_descriptor_corr_df['target'].eq(target)].sort_values('abs_r', ascending=False).head(1)
    if not row.empty:
        plot_pairs.append((target, row['descriptor'].iloc[0], float(row['pearson_r'].iloc[0])))
fig, axes = plt.subplots(1, max(1, len(plot_pairs)), figsize=(4.4 * max(1, len(plot_pairs)), 3.8), squeeze=False)
for ax, (target, desc, r) in zip(axes.ravel(), plot_pairs):
    sc = ax.scatter(cv_descriptor_df[desc], cv_descriptor_df[target], c=cv_descriptor_df['final_p_ratio'], cmap='coolwarm', s=32, alpha=0.85)
    clean = cv_descriptor_df[[desc, target]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(clean) >= 2:
        coef = np.polyfit(clean[desc], clean[target], deg=1)
        xs = np.linspace(float(clean[desc].min()), float(clean[desc].max()), 100)
        ax.plot(xs, coef[0] * xs + coef[1], color='black', lw=1, alpha=0.7)
    ax.set_title(f'{target}\nvs {desc}; r={r:.2f}')
    ax.set_xlabel(desc)
    ax.set_ylabel(target)
    ax.grid(alpha=0.25)
if plot_pairs:
    fig.colorbar(sc, ax=axes.ravel().tolist(), label='final p-ratio', fraction=0.025, pad=0.02)
plt.show()


## Exports

In [ ]:
run_dir.mkdir(parents=True, exist_ok=True)
trajectory_cv_summary_df.to_csv(run_dir / 'trajectory_cv_summary.csv', index=False)
sim_index_map_df.to_csv(run_dir / 'sim_index_map.csv', index=False)
initial_descriptor_df.to_csv(run_dir / 'initial_descriptors.csv', index=False)
cv_descriptor_df.to_csv(run_dir / 'cv_descriptor_rows.csv', index=False)
cv_descriptor_corr_df.to_csv(run_dir / 'cv_descriptor_correlations.csv', index=False)
cv_descriptor_projection_df.to_csv(run_dir / 'cv_descriptor_projection.csv', index=False)
cv_descriptor_importance_df.to_csv(run_dir / 'cv_descriptor_importance.csv', index=False)
intervention_descriptor_df.to_csv(run_dir / 'intervention_descriptors.csv', index=False)
intervention_screen_df.to_csv(run_dir / 'intervention_screen.csv', index=False)
intervention_shift_df.to_csv(run_dir / 'intervention_shifts.csv', index=False)
intervention_summary_df.to_csv(run_dir / 'intervention_summary.csv', index=False)
print(f'Wrote CVFEST-style descriptor projection outputs to {run_dir}')
